In [ ]:
# Databricks notebook source


In [ ]:
# Requirements Installation

# MAGIC %pip install pyyaml openpyxl
import sys
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
notebook_path = ctx.notebookPath().get()
NOTEBOOK_DIR = f"/Workspace{notebook_path.rsplit('/', 1)[0]}"
WORKSPACE_ROOT = NOTEBOOK_DIR.rsplit('/databricks_bundle/src', 1)[0]
FILES_ROOT = WORKSPACE_ROOT if WORKSPACE_ROOT.endswith('/files') else f"{WORKSPACE_ROOT}/files"
sys.path.insert(0, f"{FILES_ROOT}/databricks_bundle/drugdev/gold_framework/gold_engine")


In [ ]:
# Imports

from pyspark.sql import SparkSession
from datetime import datetime
import importlib.util
import sys
import os
import re
import yaml
import json
import logging
import uuid
logger = logging.getLogger(__name__)
spark = SparkSession.builder.getOrCreate()


In [ ]:
# -- Performance tuning -------------------------------------------------------

spark.conf.set("spark.sql.shuffle.partitions", "32")
spark.conf.set("spark.sql.adaptive.enabled",                          "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled",       "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled",        "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled",          "true")
spark.conf.set("spark.databricks.delta.merge.optimizeInsertOnlyMerge.enabled", "true")
spark.conf.set("spark.databricks.delta.merge.repartitionBeforeWrite.enabled",  "true")
spark.conf.set("spark.databricks.delta.merge.enableLowShuffle",       "true")
# -----------------------------------------------------------------------------

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
notebook_path = ctx.notebookPath().get()

NOTEBOOK_DIR = f"/Workspace{notebook_path.rsplit('/', 1)[0]}"
WORKSPACE_ROOT = NOTEBOOK_DIR.rsplit('/databricks_bundle/src', 1)[0]
FILES_ROOT = WORKSPACE_ROOT if WORKSPACE_ROOT.endswith('/files') else f"{WORKSPACE_ROOT}/files"
GOLD_DIR     = os.path.abspath(f"{FILES_ROOT}/databricks_bundle/drugdev/gold_framework")


print("Notebook Directory      :", NOTEBOOK_DIR)
print("Gold Framework Directory:", GOLD_DIR)


In [ ]:
# Workflow Parameters (widgets)

def _widget(name: str, default: str) -> str:
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name).strip()

CATALOG          = _widget("catalog", "dev-drugdev_da-koios-catalog")
SILVER_SCHEMA    = _widget("silver_schema", "dev_drugdev_silver")
GOLD_SCHEMA      = _widget("gold_schema", "dev_drugdev_gold")
METADATA_CATALOG = _widget("metadata_catalog", "dev-drugdev_da-koios-catalog")
REGISTRY_SCHEMA  = _widget("registry_schema", "dev_drugdev_common")
METADATA_SCHEMA  = _widget("metadata_schema", "dev_drugdev_common")
ENVIRONMENT      = _widget("environment", "dev")
PII_UNMASK_GROUP = _widget("pii_unmask_group", "pii_unmasked_access")
SOURCE_BUCKET    = _widget("source_bucket", "exelixis-clearlake-daplex-dev-us-west-2-441447966705-raw")
RUN_DATE         = _widget("run_date", datetime.now().strftime("%Y%m%d"))
PIPELINE_RUN_ID  = _widget("pipeline_run_id", uuid.uuid4().hex)

MODE             = _widget("mode", "all")     # all | object
OBJECT_NAME      = _widget("object_name", "")
MAX_WORKERS_RAW  = _widget("max_workers", "0")

default_config_path = os.path.abspath(f"{NOTEBOOK_DIR}/../../config/environments/{ENVIRONMENT}.yaml")
CONFIG_PATH = _widget("config_path", default_config_path) or default_config_path

mode          = MODE.strip().lower()
object_name   = OBJECT_NAME.strip() or None
run_date      = RUN_DATE
pipeline_run_id = PIPELINE_RUN_ID.strip() or uuid.uuid4().hex
environment   = ENVIRONMENT
source_bucket = SOURCE_BUCKET

if mode not in {"all", "object"}:
    raise ValueError("Invalid mode. Use one of: all, object")
if mode == "object" and not object_name:
    raise ValueError("OBJECT_NAME is required when MODE is object")

def _compute_dynamic_workers(_spark) -> tuple[int, int, int, int]:
    try:
        cluster_workers = int(_spark.conf.get("spark.databricks.clusterUsageTags.clusterWorkers", "0"))
    except Exception:
        cluster_workers = 0
    try:
        executor_cores = int(_spark.conf.get("spark.executor.cores", "4"))
    except Exception:
        executor_cores = 4
    if cluster_workers <= 0:
        cluster_workers = 2
    total_parallel_slots = max(cluster_workers * max(executor_cores, 1), 2)
    dynamic_workers = max(2, min(16, total_parallel_slots // 2))
    return dynamic_workers, cluster_workers, executor_cores, total_parallel_slots

dynamic_workers, active_workers, executor_cores, total_slots = _compute_dynamic_workers(spark)
try:
    max_workers = int(MAX_WORKERS_RAW)
except Exception as exc:
    raise ValueError(f"max_workers must be an integer, got: {MAX_WORKERS_RAW}") from exc
gold_workers = dynamic_workers if max_workers == 0 else max_workers

spark.conf.set("drugdev.METADATA_CATALOG",              METADATA_CATALOG)
spark.conf.set("drugdev.METADATA_SCHEMA",               METADATA_SCHEMA)
spark.conf.set("drugdev.REGISTRY_SCHEMA",               REGISTRY_SCHEMA)
spark.conf.set("drugdev.CATALOG",                       CATALOG)
spark.conf.set("drugdev.SILVER_SCHEMA",                 SILVER_SCHEMA)
spark.conf.set("drugdev.GOLD_SCHEMA",                   GOLD_SCHEMA)
spark.conf.set("drugdev.environment",                   ENVIRONMENT)
spark.conf.set("drugdev.PII_UNMASK_GROUP",              PII_UNMASK_GROUP)
spark.conf.set("drugdev.source_bucket",                 SOURCE_BUCKET)
spark.conf.set("drugdev.RUN_DATE",                      run_date)
spark.conf.set("drugdev.YML_GOLD_CONFIG_PATH",          f"{GOLD_DIR}/configs/drugdev_gold_config.yaml")
spark.conf.set("drugdev.YML_GOLD_STUDY_OVERRIDES_PATH", f"{GOLD_DIR}/configs/drugdev_gold_study_overrides.yaml")
spark.conf.set("drugdev.CONFIG_PATH",                   CONFIG_PATH)
spark.conf.set("drugdev.PIPELINE_RUN_ID",              pipeline_run_id)

METADATA_CATALOG = spark.conf.get("drugdev.METADATA_CATALOG")
METADATA_SCHEMA  = spark.conf.get("drugdev.METADATA_SCHEMA")
REGISTRY_SCHEMA  = spark.conf.get("drugdev.REGISTRY_SCHEMA")
CATALOG          = spark.conf.get("drugdev.CATALOG")
GOLD_SCHEMA      = spark.conf.get("drugdev.GOLD_SCHEMA")
environment      = spark.conf.get("drugdev.environment")

print(f"Mode               : {mode}")
print(f"Scope              : object={object_name or 'ALL'}")
print(f"Cluster workers    : {active_workers} executor(s)")
print(f"Executor cores     : {executor_cores}")
print(f"Parallel slots     : {total_slots}")
print(f"Gold workers       : {gold_workers} ({'auto' if max_workers == 0 else 'manual override'})")
print(f"RUN_DATE           : {run_date}")
print(f"PIPELINE_RUN_ID    : {pipeline_run_id}")
print(f"CATALOG            : {CATALOG}")
print(f"GOLD_SCHEMA        : {GOLD_SCHEMA}")
print(f"Environment        : {environment}")
print(f"Objects YAML path  : {spark.conf.get('drugdev.YML_GOLD_CONFIG_PATH')}")
print(f"Overrides YAML path: {spark.conf.get('drugdev.YML_GOLD_STUDY_OVERRIDES_PATH')}")


In [ ]:
# Helper Function - dynamic module loader


def import_from_path(name, path):
    module_dir = os.path.dirname(os.path.abspath(path))
    if module_dir not in sys.path:
        sys.path.insert(0, module_dir)
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    mod.dbutils = dbutils
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

def get_all_objects(spark) -> list:
    return [r.object_name for r in spark.sql(
        f"SELECT DISTINCT object_name FROM `{METADATA_CATALOG}`.{REGISTRY_SCHEMA}.gold_object_registry "
        f"WHERE is_enabled = TRUE ORDER BY object_name"
    ).collect()]

_SECRET_PATTERN = re.compile(r"\{\{SECRET:([^}]+)\}\}")

_secret_cache: dict[str, str] = {}

def _load_aws_secret() -> dict[str, str]:
    with open(CONFIG_PATH) as _f:
        _raw_cfg = yaml.safe_load(_f)
    secret_name = _raw_cfg.get("aws_secret_name")
    if not secret_name:
        raise RuntimeError("aws_secret_name is not set in environment YAML")

    try:
        import boto3
        region = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")
        client = boto3.client("secretsmanager", region_name=region)
        response = client.get_secret_value(SecretId=secret_name)
        secret_str = response.get("SecretString") or ""
        payload = json.loads(secret_str)
        logger.info("AWS secret loaded: %s (%d keys)", secret_name, len(payload))
        return payload
    except ImportError:
        logger.warning("boto3 not available; using environment fallback")
        return {}
    except Exception as exc:
        logger.warning("Could not load AWS secret %r: %s", secret_name, exc)
        return {}


def _get_secret(key: str) -> str:
    global _secret_cache
    if not _secret_cache:
        _secret_cache = _load_aws_secret()

    if key in _secret_cache:
        return _secret_cache[key]

    env_key = re.sub(r"[^A-Z0-9]", "_", f"SECRET_{key}".upper())
    val = os.environ.get(env_key)
    if val is not None:
        return val

    raise RuntimeError(f"Secret key {key!r} not found")


def _resolve_secret_tokens(value: str) -> str:
    def _replace(m: re.Match) -> str:
        return _get_secret(m.group(1))
    return _SECRET_PATTERN.sub(_replace, value)


def _resolve_secrets_in(obj):
    if isinstance(obj, str):
        return _resolve_secret_tokens(obj) if "{{SECRET:" in obj else obj
    if isinstance(obj, dict):
        return {k: _resolve_secrets_in(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_resolve_secrets_in(i) for i in obj]
    return obj


def load_config() -> dict:
    with open(CONFIG_PATH) as f:
        raw = yaml.safe_load(f)
    cfg = _resolve_secrets_in(raw)
    logger.info("Config loaded: %s  env=%s", CONFIG_PATH, cfg.get("environment"))
    return cfg


In [ ]:
# Performance Tuning (Delta / AQE optimizations)

spark.conf.set("spark.sql.shuffle.partitions",                                 "200")
spark.conf.set("spark.sql.adaptive.enabled",                                   "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled",                "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled",                 "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled",                   "true")
spark.conf.set("spark.databricks.delta.merge.optimizeInsertOnlyMerge.enabled", "true")
spark.conf.set("spark.databricks.delta.merge.repartitionBeforeWrite.enabled",  "true")
spark.conf.set("spark.databricks.delta.merge.enableLowShuffle",                "true")

print("Performance tuning applied.")


In [ ]:
# Create Gold Metadata Table (idempotent -- runs once per environment)
#
# gold_object_registry -- one row per gold object

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{REGISTRY_SCHEMA}.gold_object_registry (
    object_id         STRING      COMMENT 'UUID primary key',
    object_name       STRING      COMMENT 'Gold object name',
    object_type       STRING      COMMENT 'DIM, FACT, AGG, VIEW, TABLE',
    build_strategy    STRING      COMMENT 'PYSPARK_MODEL, FULL_REFRESH, VIEW_DDL',
    class_path        STRING      COMMENT 'Python class path for model execution',
    write_mode        STRING      COMMENT 'overwrite or append',
    execution_order   INT         COMMENT 'Execution wave ordering',
    dependencies      STRING      COMMENT 'JSON array of dependent object names',
    sql_template      STRING      COMMENT 'Optional SQL template for SQL strategies',
    partition_cols    STRING      COMMENT 'JSON array of partition columns',
    is_enabled        BOOLEAN     COMMENT 'False = exclude from pipeline runs',
    tags              STRING      COMMENT 'JSON map of tags',
    domain_name       STRING      COMMENT 'Domain label from YAML',
    data_product_name STRING      COMMENT 'Data product label from YAML',
    created_at        TIMESTAMP
) USING DELTA
COMMENT 'Gold metadata registry. One row per Gold object. Consumed by Gold Engine.'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{METADATA_SCHEMA}.pipeline_execution_metrics (
    run_id                STRING,
    job_name              STRING,
    task_name             STRING,
    domain                STRING,
    vendor                STRING,
    study_id              STRING,
    layer                 STRING,
    status                STRING,
    error_message         STRING,
    error_type            STRING,
    start_time            TIMESTAMP,
    end_time              TIMESTAMP,
    duration_seconds      INT,
    execution_timestamp   TIMESTAMP,
    execution_date        DATE,
    records_read          BIGINT,
    records_written       BIGINT,
    files_processed       INT,
    bytes_processed       BIGINT,
    checkpoint_path       STRING,
    triggered_by          STRING
) USING DELTA
COMMENT 'Pipeline execution audit table across Bronze, Silver, and Gold layers.'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{METADATA_SCHEMA}.alert_history (
    alert_id            STRING,
    alert_type          STRING,
    severity            STRING,
    title               STRING,
    message             STRING,
    domain              STRING,
    vendor              STRING,
    study_id            STRING,
    entity              STRING,
    status              STRING,
    alert_timestamp     TIMESTAMP,
    alert_date          DATE,
    resolved_at         TIMESTAMP,
    resolved_by         STRING,
    resolution_note     STRING,
    teams_sent          BOOLEAN,
    email_sent          BOOLEAN,
    created_at          TIMESTAMP
) USING DELTA
COMMENT 'Pipeline alert and notification log table for operational events.'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{METADATA_SCHEMA}.dq_validation_results_log (
    validation_id          STRING,
    validation_timestamp   TIMESTAMP,
    validation_date        DATE,
    schema_id              STRING,
    domain                 STRING,
    vendor                 STRING,
    study_id               STRING,
    layer                  STRING,
    schema_name            STRING,
    table_name             STRING,
    rule_id                STRING,
    rule_name              STRING,
    rule_type              STRING,
    column_name            STRING,
    severity               STRING,
    mode                   STRING,
    result                 STRING,
    record_count           BIGINT,
    failed_record_count    BIGINT,
    pass_rate_pct          DOUBLE,
    sample_failed_records  STRING
) USING DELTA
COMMENT 'Data quality validation results across Bronze, Silver, and Gold layers.'
""")

print("Gold metadata tables ready.")
print(f"  `{METADATA_CATALOG}`.{REGISTRY_SCHEMA}.gold_object_registry")
print(f"  `{METADATA_CATALOG}`.{METADATA_SCHEMA}.pipeline_execution_metrics")
print(f"  `{METADATA_CATALOG}`.{METADATA_SCHEMA}.alert_history")
print(f"  `{METADATA_CATALOG}`.{METADATA_SCHEMA}.dq_validation_results_log")


In [ ]:
# Load Gold Metadata from YAML -> metadata tables

gold_metadata_loader = import_from_path(
    "gold_metadata_loader",
    f"{GOLD_DIR}/metadata_service/metadata_loader.py"
)

gold_metadata_loader.main()

print("Gold metadata loaded successfully.")


In [ ]:
# Verification - inspect loaded metadata

print("\n-- gold_object_registry ------------------------------------------------")
display(spark.sql(f"""
    SELECT
        object_id,
        object_name,
        object_type,
        build_strategy,
        class_path,
        write_mode,
        execution_order,
        dependencies,
        partition_cols,
        is_enabled,
        tags,
        created_at
    FROM `{METADATA_CATALOG}`.{REGISTRY_SCHEMA}.gold_object_registry
    ORDER BY execution_order, object_name
"""))


In [ ]:
# Populate master tables inline (no child notebook execution)
# Logic copied from:
# - populate_participant_status_master.py
# - populate_site_status_master.py
# - populate_cohort_status_master.py

from delta.tables import DeltaTable
from pyspark.sql import functions as F
PLATFORM_SRC_DIR = os.path.abspath(f"{NOTEBOOK_DIR}/../../src")
if PLATFORM_SRC_DIR not in sys.path:
    sys.path.insert(0, PLATFORM_SRC_DIR)



cat = f"`{CATALOG}`"
common = f"{METADATA_SCHEMA}"


def _silver_fqn(table: str) -> str:    
    return f"{cat}.{SILVER_SCHEMA}.{table}"

print("=== Inline Master Population Start ===")

# 1) participant_status_master
participant_master_tbl = f"{cat}.{common}.participant_status_master"
irt_tbl = _silver_fqn("irt_subject_summary_report")
edc_tbl = _silver_fqn("edc_subject_summary")
ctms_tbl = _silver_fqn("ctms_study_subject")

irt_df = spark.table(irt_tbl).select("study_id", "subject", F.col("status").alias("irt_status"))
edc_df = spark.table(edc_tbl).select(F.col("study_id").alias("study_id"), F.col("participant_number").alias("subject"), F.col("subject_status").alias("edc_status"))
ctms_df = spark.table(ctms_tbl).select("study_id", F.col("participant_number").alias("subject"), F.col("subject_status").alias("ctms_status"))

new_combos = (
    irt_df.join(edc_df, ["study_id", "subject"], "left").join(ctms_df, ["study_id", "subject"], "left").select("irt_status", "edc_status", "ctms_status")
    .union(edc_df.join(irt_df, ["study_id", "subject"], "left").join(ctms_df, ["study_id", "subject"], "left").select("irt_status", "edc_status", "ctms_status"))
    .union(ctms_df.join(irt_df, ["study_id", "subject"], "left").join(edc_df, ["study_id", "subject"], "left").select("irt_status", "edc_status", "ctms_status"))
    .distinct()
    .withColumn("participant_status_normalized", F.lit(None).cast("string"))
    .withColumn("notes", F.lit(None).cast("string"))
    .withColumn("updated_by", F.current_user())
    .withColumn("updated_at", F.current_timestamp())
)

(
    DeltaTable.forName(spark, participant_master_tbl)
    .alias("t")
    .merge(
        new_combos.alias("s"),
        """
        (t.irt_status  IS NOT DISTINCT FROM s.irt_status)
        AND (t.edc_status  IS NOT DISTINCT FROM s.edc_status)
        AND (t.ctms_status IS NOT DISTINCT FROM s.ctms_status)
        """,
    )
    .whenNotMatchedInsert(values={
        "irt_status":                    "s.irt_status",
        "edc_status":                    "s.edc_status",
        "ctms_status":                   "s.ctms_status",
        "participant_status_normalized": "s.participant_status_normalized",
        "notes":                         "s.notes",
        "updated_by":                    "s.updated_by",
        "updated_at":                    "s.updated_at",
    })
    .execute()
)

participant_unmapped = spark.table(participant_master_tbl).filter(F.col("participant_status_normalized").isNull()).count()
print(f"participant_status_master populated. Unmapped rows: {participant_unmapped}")

# 2) site_status_master
site_master_tbl = f"{cat}.{common}.site_status_master"
ctms_site_tbl = _silver_fqn("ctms_study_site")

new_statuses = (
    spark.table(ctms_site_tbl)
    .select(F.col("study_id"), F.col("study_site_status").alias("ctms_status"))
    .filter(F.col("ctms_status").isNotNull())
    .distinct()
    .withColumn("group_name", F.lit(None).cast("string"))
    .withColumn("site_status_normalized", F.lit(None).cast("string"))
    .withColumn("notes", F.lit(None).cast("string"))
    .withColumn("updated_at", F.current_timestamp())
)

(
    DeltaTable.forName(spark, site_master_tbl)
    .alias("t")
    .merge(
        new_statuses.alias("s"),
        """
        (t.study_id IS NOT DISTINCT FROM s.study_id)
        AND (t.ctms_status IS NOT DISTINCT FROM s.ctms_status)
        """,
    )
    .whenNotMatchedInsert(values={
        "study_id":               "s.study_id",
        "ctms_status":            "s.ctms_status",
        "group_name":             "s.group_name",
        "site_status_normalized": "s.site_status_normalized",
        "notes":                  "s.notes",
        "updated_at":             "s.updated_at",
    })
    .execute()
)

site_unmapped = spark.table(site_master_tbl).filter(F.col("site_status_normalized").isNull()).count()
print(f"site_status_master populated. Unmapped rows: {site_unmapped}")

# 3) cohort_status_master
cohort_master_tbl = f"{cat}.{common}.cohort_status_master"
irt_cohort_src = _silver_fqn("irt_subject_summary_report")
edc_cohort_src = _silver_fqn("edc_subject_summary")

irt_cohort_df = spark.table(irt_cohort_src).filter("cohort IS NOT NULL").select("study_id", "subject", F.col("cohort").alias("irt_cohort"))
edc_cohort_df = spark.table(edc_cohort_src).filter("cohort IS NOT NULL").select(F.col("study_id").alias("study_id"), F.col("participant_number").alias("subject"), F.col("cohort").alias("edc_cohort"))

new_cohorts = (
    irt_cohort_df.join(edc_cohort_df, ["study_id", "subject"], "left").select("study_id", "irt_cohort", "edc_cohort")
    .union(edc_cohort_df.join(irt_cohort_df, ["study_id", "subject"], "left").select("study_id", "irt_cohort", "edc_cohort"))
    .distinct()
    .withColumn("cohort_normalized", F.lit(None).cast("string"))
    .withColumn("cohort_display_name", F.lit(None).cast("string"))
    .withColumn("cohort_group", F.lit(None).cast("string"))
    .withColumn("notes", F.lit(None).cast("string"))
    .withColumn("updated_at", F.current_timestamp())
)

(
    DeltaTable.forName(spark, cohort_master_tbl)
    .alias("t")
    .merge(
        new_cohorts.alias("s"),
        """
        (t.study_id IS NOT DISTINCT FROM s.study_id)
        AND (t.irt_cohort IS NOT DISTINCT FROM s.irt_cohort)
        AND (t.edc_cohort IS NOT DISTINCT FROM s.edc_cohort)
        """,
    )
    .whenNotMatchedInsert(values={
        "study_id":            "s.study_id",
        "irt_cohort":          "s.irt_cohort",
        "edc_cohort":          "s.edc_cohort",
        "cohort_normalized":   "s.cohort_normalized",
        "cohort_display_name": "s.cohort_display_name",
        "cohort_group":        "s.cohort_group",
        "notes":               "s.notes",
        "updated_at":          "s.updated_at",
    })
    .execute()
)

cohort_unmapped = spark.table(cohort_master_tbl).filter(F.col("cohort_normalized").isNull()).count()
print(f"cohort_status_master populated. Unmapped rows: {cohort_unmapped}")

print("=== Inline Master Population Complete ===")


In [ ]:
# Dependency check - required metadata lookup tables must exist before Gold execution

# REQUIRED = pipeline should fail fast when missing
required_metadata_tables = [
    "ct_portfolio_source",
    "participant_status_master",
    "milestone_code_map",
    "country_region_mapping",
    "dim_participant_phase_rules",
    "study_tf_plus_tumors",
]

# OPTIONAL = model has fallback behavior when missing
optional_metadata_tables = [
    "site_status_master",
    "cohort_status_master",
    "treatment_arm_master",
    "dim_participant_cohort_overrides",
    "cohort_summary_mapping",
    "planisware_exs_ref",
]

metadata_namespace = f"`{METADATA_CATALOG}`.{REGISTRY_SCHEMA}"


def _table_exists(tbl_name: str) -> bool:
    return spark.sql(
        f"SHOW TABLES IN {metadata_namespace} LIKE '{tbl_name}'"
    ).count() > 0


missing_required = [t for t in required_metadata_tables if not _table_exists(t)]
missing_optional = [t for t in optional_metadata_tables if not _table_exists(t)]

if missing_required:
    missing_csv = ", ".join(missing_required)
    raise RuntimeError(
        "Gold dependency check failed. Missing required metadata table(s) "
        f"in {metadata_namespace}: {missing_csv}"
    )

print("Gold dependency check passed for required metadata tables:")
for table_name in required_metadata_tables:
    print(f"  - {metadata_namespace}.{table_name}")

if missing_optional:
    print("\nOptional metadata tables not found (fallback logic will be used where applicable):")
    for table_name in missing_optional:
        print(f"  - {metadata_namespace}.{table_name}")
else:
    print("\nAll optional metadata tables are present.")


In [ ]:
# Run Gold Pipeline
#
# Runs all enabled objects by execution wave, or a single object when mode=object.

def _as_bool(v: str) -> bool:
    return str(v).strip().lower() in {"1", "true", "yes", "y", "on"}


def _get_conf_required(key: str) -> str:
    value = spark.conf.get(key, "").strip()
    if not value:
        raise ValueError(f"Missing required Spark conf: {key}")
    return value


def _get_conf_optional(key: str, default: str = "") -> str:
    return spark.conf.get(key, default).strip()


def _json_list_from_conf(key: str, required: bool = False) -> list:
    raw = _get_conf_required(key) if required else _get_conf_optional(key, "[]")
    try:
        parsed = json.loads(raw) if raw else []
    except Exception as exc:
        raise ValueError(f"Spark conf {key} must be valid JSON list") from exc
    if not isinstance(parsed, list):
        raise ValueError(f"Spark conf {key} must be a JSON list")
    return parsed


def _json_dict_from_conf(key: str, default: str = "{}") -> dict:
    raw = _get_conf_optional(key, default)
    try:
        parsed = json.loads(raw) if raw else {}
    except Exception as exc:
        raise ValueError(f"Spark conf {key} must be valid JSON object") from exc
    if not isinstance(parsed, dict):
        raise ValueError(f"Spark conf {key} must be a JSON object")
    return parsed


def _build_alerting_config_from_spark() -> dict:
    email_enabled = _as_bool(_get_conf_optional("drugdev.NOTIFY_EMAIL_ENABLED", "false"))
    teams_enabled = _as_bool(_get_conf_optional("drugdev.NOTIFY_TEAMS_ENABLED", "false"))

    missing = []
    if email_enabled:
        for k in [
            "drugdev.NOTIFY_EMAIL_FROM",
            "drugdev.NOTIFY_EMAIL_REPLY_TO",
            "drugdev.NOTIFY_EMAIL_RECIPIENTS_CRITICAL_JSON",
            "drugdev.NOTIFY_EMAIL_RECIPIENTS_HIGH_JSON",
            "drugdev.NOTIFY_EMAIL_RECIPIENTS_MEDIUM_JSON",
        ]:
            if not spark.conf.get(k, "").strip():
                missing.append(k)
    if teams_enabled and not spark.conf.get("drugdev.NOTIFY_TEAMS_WEBHOOK_URL", "").strip():
        missing.append("drugdev.NOTIFY_TEAMS_WEBHOOK_URL")
    if missing:
        raise ValueError("Missing required Spark conf(s) for alerting: " + ", ".join(missing))

    email_cfg = {
        "enabled": email_enabled,
        "smtp_driver": _get_conf_optional("drugdev.NOTIFY_EMAIL_DRIVER", "ses"),
        "from_address": _get_conf_optional("drugdev.NOTIFY_EMAIL_FROM"),
        "reply_to": _get_conf_optional("drugdev.NOTIFY_EMAIL_REPLY_TO"),
        "ses_region": _get_conf_optional("drugdev.NOTIFY_EMAIL_SES_REGION", _get_conf_optional("AWS_DEFAULT_REGION", "us-west-2")),
        "recipients": {
            "critical": _json_list_from_conf("drugdev.NOTIFY_EMAIL_RECIPIENTS_CRITICAL_JSON") if email_enabled else [],
            "high": _json_list_from_conf("drugdev.NOTIFY_EMAIL_RECIPIENTS_HIGH_JSON") if email_enabled else [],
            "medium": _json_list_from_conf("drugdev.NOTIFY_EMAIL_RECIPIENTS_MEDIUM_JSON") if email_enabled else [],
            "low": _json_list_from_conf("drugdev.NOTIFY_EMAIL_RECIPIENTS_LOW_JSON"),
        },
    }

    teams_cfg = {
        "enabled": teams_enabled,
        "webhook_url": _get_conf_optional("drugdev.NOTIFY_TEAMS_WEBHOOK_URL"),
    }

    routing_cfg = _json_dict_from_conf(
        "drugdev.NOTIFY_ROUTING_JSON",
        '{"CRITICAL":{"teams":true},"HIGH":{"teams":true},"MEDIUM":{"teams":false},"LOW":{"teams":false}}',
    )
    domain_recipients_cfg = _json_dict_from_conf("drugdev.NOTIFY_DOMAIN_RECIPIENTS_JSON", "{}")

    return {
        "notifications": {
            "email": email_cfg,
            "teams": teams_cfg,
            "routing": routing_cfg,
            "domain_recipients": domain_recipients_cfg,
        }
    }


if mode == "all":
    run_plan = [("all", None)]
    objects = get_all_objects(spark)
    print(f"=== Gold Pipeline: {len(objects)} active object(s) queued ===")
else:
    run_plan = [("object", object_name)]
    print(f"=== Gold Pipeline: single object run: {object_name} ===")

gold_engine = import_from_path(
    "gold_engine",
    f"{GOLD_DIR}/gold_engine/gold_engine.py"
)

# AlertNotifier requires these conf keys; set explicit workflow values.
if not spark.conf.get("drugdev.BRONZE_SCHEMA", "").strip():
    spark.conf.set("drugdev.BRONZE_SCHEMA", f"{environment}_drugdev_bronze")
if not spark.conf.get("drugdev.CONFIG_PATH", "").strip():
    spark.conf.set("drugdev.CONFIG_PATH", CONFIG_PATH)

SILVER_ENGINE_DIR = os.path.abspath(f"{FILES_ROOT}/databricks_bundle/drugdev/silver_framework/silver_engine")
NOTIFICATION_DIR = os.path.abspath(f"{FILES_ROOT}/databricks_bundle/drugdev/notification_faramework")
if SILVER_ENGINE_DIR not in sys.path:
    sys.path.insert(0, SILVER_ENGINE_DIR)

alert_notifier = import_from_path(
    "alert_notifier",
    f"{NOTIFICATION_DIR}/alert_notifier.py"
)
cfg = _build_alerting_config_from_spark()
notifier = alert_notifier.AlertNotifier(cfg)

all_results = []
failed_runs = []

for run_mode, run_object in run_plan:
    label = "ALL" if run_mode == "all" else run_object
    print(f"\n--- Running Gold for: {label} ---")
    try:
        gold_engine.main(mode=run_mode, object_name=run_object, max_workers=gold_workers)
        all_results.append({"target": label, "status": "SUCCESS"})
    except Exception as exc:
        logging.error("Gold failed for %s: %s", label, exc, exc_info=True)
        failed_runs.append(f"{label}: {exc}")
        all_results.append({"target": label, "status": "FAILED", "error": str(exc)})

print("\n=== Gold Results ===")
for r in all_results:
    icon = 'S' if r.get('status') == 'SUCCESS' else 'F'
    print(f"  {icon} {r.get('target')}: {r.get('status')}")

if failed_runs:
    msg = (
        f"Gold pipeline failed for {len(failed_runs)} run target(s).\n"
        f"Details:\n" + "\n".join(f"  - {d}" for d in failed_runs)
    )
    try:
        notifier.send_alert(
            alert_type='GOLD_COMPLETION',
            severity='CRITICAL',
            title=f"Gold Completion [{environment}] - FAILED",
            message=msg[:400],
            context={
                'mode': mode,
                'object_name': object_name or 'ALL',
                'failed_runs': len(failed_runs),
                'run_date': run_date,
            },
        )
    except Exception as alert_exc:
        logging.warning("Failed to send gold failure alert: %s", alert_exc)
    raise RuntimeError(msg)


In [ ]:
# Run Summary - current row counts per active Gold object

object_filter = f"AND object_name = '{object_name}'" if object_name else ""

active_objects = spark.sql(f"""
    SELECT DISTINCT object_name, object_type, build_strategy
    FROM `{METADATA_CATALOG}`.{REGISTRY_SCHEMA}.gold_object_registry
    WHERE is_enabled = TRUE
      {object_filter}
    ORDER BY object_name
""").collect()

summary_rows = []
for row in active_objects:
    try:
        gold_tbl = f"`{CATALOG}`.{GOLD_SCHEMA}.{row['object_name']}"
        if row['build_strategy'] == 'VIEW_DDL':
            current_count = 'VIEW'
        else:
            current_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {gold_tbl}").collect()[0]['cnt']
        summary_rows.append({
            "object_name":   row["object_name"],
            "object_type":   row["object_type"],
            "build_strategy": row["build_strategy"],
            "row_count":     current_count,
        })
    except Exception as e:
        summary_rows.append({
            "object_name":   row["object_name"],
            "object_type":   row["object_type"],
            "build_strategy": row["build_strategy"],
            "row_count":     f"ERROR: {e}"
        })

if summary_rows:
    summary_df = spark.createDataFrame(summary_rows)
    print("\n-- Gold Run Summary ----------------------------------------------------")
    display(summary_df)
else:
    print("No active Gold objects found in registry.")
